# Fase 3 — Regras de Associação: Candidatos 2026 (TSE)

## Pergunta de negócio

> **Que combinações de características dos candidatos aparecem juntas com mais frequência do que se fossem independentes? Em particular: quais atributos (partido, gênero, escolaridade, cor/raça, UF de nascimento) estão associados a cada perfil identificado na Fase 2 — e, assim que a eleição ocorrer, a ser eleito?**

Até aqui, a clusterização (Fase 2) agrupou candidatos por **distância** em variáveis numéricas (idade, escolaridade, patrimônio). Regras de associação fazem uma pergunta diferente: em vez de "quem está perto de quem", **quais combinações de atributos categóricos aparecem juntas com mais frequência do que o acaso explicaria**? A técnica clássica pra isso é o **Apriori**, que originalmente resolve o problema da "cesta de compras" (que produtos são comprados juntos) — aqui, cada candidato é uma cesta, e cada valor de cada variável categórica (partido, gênero, escolaridade...) é um item dessa cesta.


## Configuração

Use os **mesmos valores** que você usou nas Fases 0/1/2, para carregar o arquivo certo.


In [47]:
CARGO = "DEPUTADO FEDERAL"  # mesmo valor usado nas Fases 0/1/2
UF = ["DF", "GO", "MT", "MS"]  # mesmo valor usado nas Fases 0/1/2
ANO_ELEICAO = 2026

## Preparando a base — reconstituindo os clusters nomeados da Fase 2

Reproduzimos a Parte 2 de `02_clusterizacao.ipynb`: mesmas variáveis (`IDADE`, `ANOS_ESTUDO`, `Total_Bens_Log`), filtro de patrimônio por IQR, `MinMaxScaler`, semente 42 e **Bisecting K-Means com k=6, critério de inércia total**. A cada passo, dividimos o grupo com maior soma das distâncias ao quadrado ao próprio centro; não usamos a inércia média por candidato. Os rótulos A–F seguem a ordem decrescente de tamanho, como na Fase 2.

Os nomes resumem os radares e a ficha técnica dessa partição:

| Cluster | Nome                                          | Candidatos | Idade média | Anos de estudo médios | Patrimônio mediano |
| ------- | --------------------------------------------- | ---------: | ----------: | --------------------: | -----------------: |
| A       | Adultos com superior e bens                   |        203 |        42,4 |                  16,0 |         R$ 495.000 |
| B       | Mais velhos com superior e bens               |        184 |        61,6 |                  15,8 |         R$ 895.668 |
| C       | Alta escolaridade sem bens declarados         |        106 |        49,5 |                  14,8 |               R$ 0 |
| D       | Jovens com menor escolaridade e poucos bens   |         90 |        36,7 |                   9,7 |               R$ 0 |
| E       | Adultos com escolaridade intermediária e bens |         75 |        40,4 |                  11,2 |         R$ 217.000 |
| F       | Mais velhos com menor escolaridade e bens     |         34 |        61,3 |                   9,5 |         R$ 496.104 |

A e B se distinguem principalmente pela idade; C tem escolaridade alta, mas todos os registros de bens são zero; D reúne os mais jovens e tem poucos bens declarados; E e F têm escolaridade menor que A e B e se distinguem pela idade. Os termos são relativos a este recorte. O radar mostra médias padronizadas do **logaritmo dos bens**, não valores em reais nem percentuais de riqueza. Os nomes não indicam renda, profissão ou experiência política.

Calculamos a partição aqui para que o notebook rode de forma independente. Se alterar a base, a semente, o critério ou `k`, revise os nomes e os números pela ficha técnica.


In [48]:
import warnings

warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from mlxtend.frequent_patterns import apriori, association_rules

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option(
    "display.max_colwidth", None
)  # não truncar antecedents/consequents nas tabelas

SEMENTE = 42
np.random.seed(
    SEMENTE
)  # mesmo racional de reprodutibilidade defensiva dos notebooks anteriores

nome_uf = (
    UF
    if isinstance(UF, str)
    else (
        "CENTRO_OESTE"
        if UF and set(UF) == {"DF", "GO", "MT", "MS"}
        else "_".join(sorted(UF)) if UF else "BRASIL"
    )
)
nome_cargo = CARGO.replace(" ", "_")
# Ajuste o nome do arquivo conforme o gerado pelo 00_preparacao_dados.ipynb
arquivo = f"dados/candidatos_{nome_cargo}_{nome_uf}_{ANO_ELEICAO}.csv"

df = pd.read_csv(arquivo)
print(f"Carregado: {arquivo}  ->  {df.shape[0]} candidatos, {df.shape[1]} colunas")
df.head()

Carregado: dados/candidatos_DEPUTADO_FEDERAL_CENTRO_OESTE_2026.csv  ->  692 candidatos, 59 colunas


,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,NR_TURNO,CD_ELEICAO,DS_ELEICAO,DT_ELEICAO,TP_ABRANGENCIA,SG_UF,SG_UE,NM_UE,CD_CARGO,DS_CARGO,SQ_CANDIDATO,NR_CANDIDATO,NM_CANDIDATO,NM_URNA_CANDIDATO,NM_SOCIAL_CANDIDATO,NR_CPF_CANDIDATO,DS_EMAIL,CD_SITUACAO_CANDIDATURA,DS_SITUACAO_CANDIDATURA,TP_AGREMIACAO,NR_PARTIDO,SG_PARTIDO,NM_PARTIDO,NR_FEDERACAO,NM_FEDERACAO,SG_FEDERACAO,DS_COMPOSICAO_FEDERACAO,SQ_COLIGACAO,NM_COLIGACAO,DS_COMPOSICAO_COLIGACAO,SG_UF_NASCIMENTO,DT_NASCIMENTO,NR_TITULO_ELEITORAL_CANDIDATO,CD_GENERO,DS_GENERO,CD_GRAU_INSTRUCAO,DS_GRAU_INSTRUCAO,CD_ESTADO_CIVIL,DS_ESTADO_CIVIL,CD_COR_RACA,DS_COR_RACA,CD_OCUPACAO,DS_OCUPACAO,CD_SIT_TOT_TURNO,DS_SIT_TOT_TURNO,BENS IMÓVEIS,BENS MÓVEIS,DINHEIRO,INVESTIMENTOS RENDA FIXA,INVESTIMENTOS RENDA VARIÁVEL,OUTROS BENS,Total_Bens,Total_Bens_Log,IDADE
0,08/09/2026,16:30:10,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,2026-10-04,ESTADUAL,DF,DF,DISTRITO FEDERAL,6,DEPUTADO FEDERAL,70002532001,3022,ALESSANDRA ALMEIDA CHIANELLI DUTRA,ALE CHIANELLI,#NULO,58442472134,NÃO DIVULGÁVEL,-3,#NE,PARTIDO ISOLADO,30,NOVO,PARTIDO NOVO,-1,#NULO,#NULO,#NULO,70001800084,PARTIDO ISOLADO,NOVO,DF,1972-11-28,11236642070,4,FEMININO,8,SUPERIOR COMPLETO,9,DIVORCIADO(A),1,BRANCA,171,JORNALISTA E REDATOR,-1,#NULO,0.0,130000.0,0.0,0.0,0.0,0.0,130000.0,11.775297,53
1,08/09/2026,16:30:10,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,2026-10-04,ESTADUAL,DF,DF,DISTRITO FEDERAL,6,DEPUTADO FEDERAL,70002532002,3012,PAULO CESAR ECHEBARRIA DE CARVALHO,PAULO ECHEBARRIA,#NULO,37959077134,NÃO DIVULGÁVEL,-3,#NE,PARTIDO ISOLADO,30,NOVO,PARTIDO NOVO,-1,#NULO,#NULO,#NULO,70001800084,PARTIDO ISOLADO,NOVO,GO,1965-10-20,7033782089,2,MASCULINO,8,SUPERIOR COMPLETO,3,CASADO(A),1,BRANCA,171,JORNALISTA E REDATOR,-1,#NULO,0.0,380000.0,0.0,0.0,0.0,15000000.0,15380000.0,16.548579,60
2,08/09/2026,16:30:10,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,2026-10-04,ESTADUAL,DF,DF,DISTRITO FEDERAL,6,DEPUTADO FEDERAL,70002532003,3004,FLAVIO HENRIQUE TORMINN FLEURY,FLAVIO FLEURY,#NULO,88847560144,NÃO DIVULGÁVEL,-3,#NE,PARTIDO ISOLADO,30,NOVO,PARTIDO NOVO,-1,#NULO,#NULO,#NULO,70001800084,PARTIDO ISOLADO,NOVO,GO,1981-09-01,46935651074,2,MASCULINO,8,SUPERIOR COMPLETO,3,CASADO(A),1,BRANCA,114,FISIOTERAPEUTA E TERAPEUTA OCUPACIONAL,-1,#NULO,0.0,53000.0,0.0,0.0,0.0,0.0,53000.0,10.878066,45
3,08/09/2026,16:30:10,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,2026-10-04,ESTADUAL,DF,DF,DISTRITO FEDERAL,6,DEPUTADO FEDERAL,70002532004,3030,CARINE TEIXEIRA MARQUES,CARINE TEIXEIRA,#NULO,1642767182,NÃO DIVULGÁVEL,-3,#NE,PARTIDO ISOLADO,30,NOVO,PARTIDO NOVO,-1,#NULO,#NULO,#NULO,70001800084,PARTIDO ISOLADO,NOVO,GO,1986-08-25,52572671066,4,FEMININO,8,SUPERIOR COMPLETO,3,CASADO(A),2,PRETA,257,EMPRESÁRIO,-1,#NULO,0.0,0.0,0.0,0.0,120000.0,0.0,120000.0,11.695255,40
4,08/09/2026,16:30:10,2026,2,ELEIÇÃO ORDINÁRIA,1,6259,Eleições Gerais Estaduais 2026,2026-10-04,ESTADUAL,DF,DF,DISTRITO FEDERAL,6,DEPUTADO FEDERAL,70002532005,3010,EMILIO KERBER FILHO,EMILIO KERBER,#NULO,92759130053,NÃO DIVULGÁVEL,-3,#NE,PARTIDO ISOLADO,30,NOVO,PARTIDO NOVO,-1,#NULO,#NULO,#NULO,70001800084,PARTIDO ISOLADO,NOVO,PR,1976-08-31,263044410167,2,MASCULINO,8,SUPERIOR COMPLETO,3,CASADO(A),1,BRANCA,185,ESCRITOR E CRÍTICO,-1,#NULO,1380000.0,0.0,0.0,0.0,0.0,0.0,1380000.0,14.137595,50


In [49]:
ANOS_ESTUDO_POR_GRAU = {
    "ANALFABETO": 0,
    "LÊ E ESCREVE": 1,
    "ENSINO FUNDAMENTAL INCOMPLETO": 4,
    "ENSINO FUNDAMENTAL COMPLETO": 8,
    "ENSINO MÉDIO INCOMPLETO": 9,
    "ENSINO MÉDIO COMPLETO": 11,
    "SUPERIOR INCOMPLETO": 13,
    "SUPERIOR COMPLETO": 16,
    # 'NÃO DIVULGÁVEL' fica de fora de propósito — mesma decisão da Versão 2 do notebook de clusterização
}

df["ANOS_ESTUDO"] = df["DS_GRAU_INSTRUCAO"].map(ANOS_ESTUDO_POR_GRAU)

In [50]:
colunas_driver = ["IDADE", "ANOS_ESTUDO", "Total_Bens_Log"]

Q1, Q3 = df["Total_Bens_Log"].quantile([0.25, 0.75])
IQR = Q3 - Q1
limite_superior = Q3 + 1.5 * IQR

df_cluster = df[
    (df["Total_Bens_Log"] <= limite_superior) & (df["ANOS_ESTUDO"].notna())
].copy()
print(
    f"Base para clusterização/regras: {df_cluster.shape[0]} candidatos "
    f"(descartados {df.shape[0] - df_cluster.shape[0]} por patrimônio extremo ou escolaridade não divulgada)"
)

scaler = MinMaxScaler()
X = scaler.fit_transform(df_cluster[colunas_driver])

Base para clusterização/regras: 692 candidatos (descartados 0 por patrimônio extremo ou escolaridade não divulgada)


In [ ]:
def bisecting_kmeans_ate_k(X, k_max, criterio="inercia", random_state=SEMENTE):
    """Reproduz as divisões da Parte 2 da Fase 2 e devolve a partição final."""

    def inercia(indices):
        if len(indices) < 2:
            return 0.0
        pontos = X[indices]
        centro = pontos.mean(axis=0)
        return float(((pontos - centro) ** 2).sum())

    clusters = {0: np.arange(X.shape[0])}
    proximo_id = 1
    for _ in range(1, k_max):
        if criterio == "tamanho":
            id_escolhido = max(clusters, key=lambda cid: len(clusters[cid]))
        elif criterio == "inercia":
            id_escolhido = max(clusters, key=lambda cid: inercia(clusters[cid]))
        else:
            raise ValueError("criterio deve ser 'tamanho' ou 'inercia'")
        indices_pai = clusters[id_escolhido]
        if len(indices_pai) < 2:
            break
        km2 = KMeans(n_clusters=2, random_state=random_state, n_init=10)
        labels2 = km2.fit_predict(X[indices_pai])
        id_a, id_b = proximo_id, proximo_id + 1
        proximo_id += 2
        del clusters[id_escolhido]
        clusters[id_a] = indices_pai[labels2 == 0]
        clusters[id_b] = indices_pai[labels2 == 1]
    return clusters


k_bisecting = 6  # mesmo k escolhido no 02_clusterizacao.ipynb (Parte 2)

criterio_bisecting = "inercia"  # mesma execução da Parte 2 da Fase 2
clusters_finais = bisecting_kmeans_ate_k(X, k_bisecting, criterio=criterio_bisecting)
ids_por_tamanho = sorted(
    clusters_finais, key=lambda cid: len(clusters_finais[cid]), reverse=True
)

rotulos = np.empty(X.shape[0], dtype=object)
for letra, cid in zip(
    [chr(65 + i) for i in range(len(ids_por_tamanho))], ids_por_tamanho
):
    rotulos[clusters_finais[cid]] = letra
df_cluster["cluster_bisecting"] = rotulos

# Mesmos nomes usados na Parte 2 da Fase 2.
NOMES_CLUSTER_BISECTING = {
    "A": "Adultos com superior e bens",
    "B": "Mais velhos com superior e bens",
    "C": "Alta escolaridade sem bens declarados",
    "D": "Jovens com menor escolaridade e poucos bens",
    "E": "Adultos com escolaridade intermediária e bens",
    "F": "Mais velhos com menor escolaridade e bens",
}
df_cluster["cluster"] = df_cluster["cluster_bisecting"].map(NOMES_CLUSTER_BISECTING)

if df_cluster["cluster"].isna().any():
    raise ValueError("Há clusters sem nome. Revise NOMES_CLUSTER_BISECTING.")

resumo_clusters = (
    df_cluster.groupby("cluster_bisecting")
    .agg(
        qtd_candidatos=("SQ_CANDIDATO", "count"),
        idade_media=("IDADE", "mean"),
        anos_estudo_medio=("ANOS_ESTUDO", "mean"),
        patrimonio_mediano=("Total_Bens", "median"),
    )
    .round(1)
)
resumo_clusters.insert(0, "nome", resumo_clusters.index.map(NOMES_CLUSTER_BISECTING))
display(resumo_clusters)

,nome,qtd_candidatos,idade_media,anos_estudo_medio,patrimonio_mediano
cluster_bisecting,,,,,
A,Adultos com superior e bens,203,42.4,16.0,495000.0
B,Mais velhos com superior e bens,184,61.6,15.8,895668.0
C,Alta escolaridade sem bens declarados,106,49.5,14.8,0.0
D,Jovens com menor escolaridade e poucos bens,90,36.7,9.7,0.0
E,Adultos com escolaridade intermediária e bens,75,40.4,11.2,217000.0
F,Mais velhos com menor escolaridade e bens,34,61.3,9.5,496103.9


## Montando a "cesta" de itens

Apriori não trabalha com números contínuos como o K-Means — ele precisa de **itens binários** (presente/ausente), do mesmo jeito que "pão" e "leite" são itens numa cesta de supermercado. Cada linha vira uma "cesta" (um candidato), e cada valor de cada variável categórica vira um "item" via one-hot encoding (`SG_PARTIDO=PT`, `DS_GENERO=FEMININO`, `cluster=Adultos com superior e bens`, ...).

Variáveis escolhidas:

- `SG_PARTIDO`, `DS_GENERO`, `DS_GRAU_INSTRUCAO`, `DS_ESTADO_CIVIL`, `DS_COR_RACA`, `SG_UF_NASCIMENTO` — perfil do candidato.
- `cluster` — o nome do cluster da Fase 2 (Bisecting K-Means, k=6, critério de inércia total), recalculado acima. Incluir o cluster como item deixa a pergunta "o que caracteriza cada cluster?" (já respondida via ficha técnica na Fase 2) ser respondida de novo do ponto de vista de regras — e permite achar combinações de 2+ variáveis que sozinhas não bastariam.
- `SG_UF` — entra quando há mais de uma UF na base, inclusive no recorte Centro-Oeste; com uma única UF ela não discrimina candidatos.
- `DS_SIT_TOT_TURNO` (eleito ou não) — só entra depois que a eleição de fato ocorrer (a coluna hoje vem toda com o mesmo valor, `#NULO`). A seção que usa isso fica pronta, mas roda condicionalmente lá na frente.

`DS_OCUPACAO` fica de fora por ora: pode ter muitas categorias com poucos candidatos e suporte insuficiente para a análise. Fica como exercício: agrupem em categorias maiores (ex.: "política", "direito", "saúde", "educação", "empresário"...) e testem.


In [52]:
colunas_apriori = [
    "SG_PARTIDO",
    "DS_GENERO",
    "DS_GRAU_INSTRUCAO",
    "DS_ESTADO_CIVIL",
    "DS_COR_RACA",
    "SG_UF_NASCIMENTO",
    "cluster",
]

if df_cluster["SG_UF"].nunique() > 1:
    colunas_apriori.append("SG_UF")

# a eleição de 2026 ainda não ocorreu -> DS_SIT_TOT_TURNO vem toda com o mesmo valor por enquanto
usar_resultado_eleicao = df_cluster["DS_SIT_TOT_TURNO"].nunique() > 1
if usar_resultado_eleicao:
    colunas_apriori.append("DS_SIT_TOT_TURNO")

cesta_base = df_cluster[["SQ_CANDIDATO"] + colunas_apriori].copy()
cesta = pd.get_dummies(cesta_base, columns=colunas_apriori, prefix_sep="=")
cesta = cesta.set_index("SQ_CANDIDATO")

print(
    f"{cesta.shape[0]} candidatos (cestas) x {cesta.shape[1]} itens possíveis (um por valor de cada variável)"
)
cesta_base.head()

692 candidatos (cestas) x 82 itens possíveis (um por valor de cada variável)


,SQ_CANDIDATO,SG_PARTIDO,DS_GENERO,DS_GRAU_INSTRUCAO,DS_ESTADO_CIVIL,DS_COR_RACA,SG_UF_NASCIMENTO,cluster,SG_UF
0,70002532001,NOVO,FEMININO,SUPERIOR COMPLETO,DIVORCIADO(A),BRANCA,DF,Mais velhos com superior e bens,DF
1,70002532002,NOVO,MASCULINO,SUPERIOR COMPLETO,CASADO(A),BRANCA,GO,Mais velhos com superior e bens,DF
2,70002532003,NOVO,MASCULINO,SUPERIOR COMPLETO,CASADO(A),BRANCA,GO,Adultos com superior e bens,DF
3,70002532004,NOVO,FEMININO,SUPERIOR COMPLETO,CASADO(A),PRETA,GO,Adultos com superior e bens,DF
4,70002532005,NOVO,MASCULINO,SUPERIOR COMPLETO,CASADO(A),BRANCA,PR,Adultos com superior e bens,DF


## Itens frequentes: o que aparece sozinho, com que frequência

O Apriori funciona em duas etapas. Primeiro, encontra **itemsets frequentes** — combinações de 1, 2, 3... itens que aparecem juntas em pelo menos `min_suporte` das cestas. Só depois, na segunda etapa, ele vira **regras** (`A → B`). Começamos pelos itemsets de um item só — a frequência "crua" de cada valor, antes de cruzar qualquer coisa.


In [53]:
min_suporte = 0.02  # mínimo de ceil(min_suporte * número de cestas) candidatos; ajuste se quiser regras mais raras (menor) ou mais comuns (maior)

itens_frequentes = apriori(cesta, min_support=min_suporte, use_colnames=True)
print(f"{len(itens_frequentes)} itemsets frequentes (min_suporte={min_suporte})")

individuais = itens_frequentes[
    itens_frequentes["itemsets"].apply(lambda x: len(x) == 1)
].copy()
individuais["item"] = individuais["itemsets"].apply(lambda x: next(iter(x)))
individuais["qtd_candidatos"] = (
    (individuais["support"] * len(cesta)).round().astype(int)
)
individuais.sort_values("support", ascending=False)[
    ["item", "support", "qtd_candidatos"]
].head(15)

1338 itemsets frequentes (min_suporte=0.02)


,item,support,qtd_candidatos
26,DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,0.651734,451
21,DS_GENERO=MASCULINO,0.611272,423
31,DS_COR_RACA=BRANCA,0.544798,377
28,DS_ESTADO_CIVIL=CASADO(A),0.518786,359
20,DS_GENERO=FEMININO,0.388728,269
52,SG_UF=GO,0.378613,262
33,DS_COR_RACA=PARDA,0.333815,231
30,DS_ESTADO_CIVIL=SOLTEIRO(A),0.315029,218
46,cluster=Adultos com superior e bens,0.293353,203
37,SG_UF_NASCIMENTO=GO,0.271676,188


### Suporte, confiança e lift — o que cada métrica mede

Regras de associação têm o formato **"se A, então B"** (A = antecedente, B = consequente), e três números resumem o quão forte é essa relação:

- **Suporte** — fração de candidatos em que A e B aparecem juntos: `suporte(A→B) = P(A e B)`. Mede o quão comum é a combinação; suporte baixo (poucos candidatos) é terreno fértil pra coincidência.
- **Confiança** — entre os candidatos que têm A, qual fração também tem B: `confiança(A→B) = P(B | A) = suporte(A e B) / suporte(A)`. É a "taxa de acerto" da regra.
- **Lift** — o quanto a confiança da regra é maior (ou menor) do que se A e B fossem independentes: `lift(A→B) = confiança(A→B) / suporte(B)`. `lift = 1` → A e B são independentes (a regra não diz nada); `lift > 1` → aparecem juntos mais do que o esperado ao acaso; `lift < 1` → aparecem juntos menos do que o esperado.

**Um cuidado importante nesta base:** algumas categorias e combinações têm poucos candidatos, mesmo com 692 registros no recorte atual. Com grupos tão pequenos, bastam 3 ou 4 coincidências pra um lift parecer altíssimo, sem que isso signifique relação real nenhuma. Por isso toda tabela de regras abaixo vem com uma coluna `qtd_candidatos` — sempre olhem ela antes de confiar num lift alto.


In [54]:
regras = association_rules(itens_frequentes, metric="lift", min_threshold=1)
regras["qtd_candidatos"] = (regras["support"] * len(cesta)).round().astype(int)
regras = regras.sort_values("confidence", ascending=False)


def formatar_regras(df_regras):
    """Troca antecedents/consequents (frozenset) por texto legível, só pra exibição —
    os dados originais (usados pra filtrar/ordenar) continuam intactos."""
    df_fmt = df_regras.copy()
    df_fmt["antecedents"] = df_fmt["antecedents"].apply(
        lambda x: ", ".join(sorted(str(i) for i in x))
    )
    df_fmt["consequents"] = df_fmt["consequents"].apply(
        lambda x: ", ".join(sorted(str(i) for i in x))
    )
    return df_fmt


print(f"{len(regras)} regras (lift >= 1)")
formatar_regras(
    regras[
        [
            "antecedents",
            "consequents",
            "qtd_candidatos",
            "support",
            "confidence",
            "lift",
        ]
    ].head(10)
)

9716 regras (lift >= 1)


,antecedents,consequents,qtd_candidatos,support,confidence,lift
3033,"DS_GENERO=FEMININO, SG_UF=GO, cluster=Adultos com superior e bens",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,32,0.046243,1.0,1.534368
6890,"DS_ESTADO_CIVIL=CASADO(A), DS_GENERO=MASCULINO, SG_UF=MT, cluster=Adultos com superior e bens",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,21,0.030347,1.0,1.534368
8452,"DS_COR_RACA=BRANCA, SG_UF=GO, SG_UF_NASCIMENTO=GO, cluster=Adultos com superior e bens",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,34,0.049133,1.0,1.534368
5932,"DS_COR_RACA=PARDA, DS_ESTADO_CIVIL=SOLTEIRO(A), SG_UF_NASCIMENTO=GO",SG_UF=GO,17,0.024566,1.0,2.641221
7366,"DS_COR_RACA=PARDA, DS_GENERO=MASCULINO, SG_UF=GO, cluster=Adultos com superior e bens",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,17,0.024566,1.0,1.534368
2648,"DS_ESTADO_CIVIL=CASADO(A), SG_PARTIDO=PL, cluster=Adultos com superior e bens",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,15,0.021676,1.0,1.534368
8665,"DS_COR_RACA=BRANCA, DS_ESTADO_CIVIL=CASADO(A), DS_GENERO=MASCULINO, SG_UF_NASCIMENTO=GO, cluster=Mais velhos com superior e bens",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,18,0.026012,1.0,1.534368
1863,"SG_UF_NASCIMENTO=MT, cluster=Adultos com superior e bens",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,30,0.043353,1.0,1.534368
5957,"DS_ESTADO_CIVIL=SOLTEIRO(A), SG_UF_NASCIMENTO=GO, cluster=Adultos com superior e bens",SG_UF=GO,15,0.021676,1.0,2.641221
5942,"DS_COR_RACA=PARDA, DS_ESTADO_CIVIL=SOLTEIRO(A), SG_UF_NASCIMENTO=MT",SG_UF=MT,14,0.020231,1.0,5.014493


### Cuidado: suporte pequeno infla o lift

Vamos ver isso na prática: as regras de maior lift da base inteira, sem nenhum filtro de suporte, comparadas às regras de maior lift **exigindo pelo menos 10 candidatos** por trás delas.


In [55]:
regras_1_consequente = regras[regras["consequents"].apply(lambda x: len(x) == 1)].copy()
colunas_exibir = ["antecedents", "consequents", "qtd_candidatos", "confidence", "lift"]

print("--- Maior lift, sem filtro de suporte ---")
display(
    formatar_regras(
        regras_1_consequente.sort_values("lift", ascending=False)[colunas_exibir].head(
            5
        )
    )
)

print("--- Maior lift, exigindo qtd_candidatos >= 10 ---")
display(
    formatar_regras(
        regras_1_consequente[regras_1_consequente["qtd_candidatos"] >= 10]
        .sort_values("lift", ascending=False)[colunas_exibir]
        .head(5)
    )
)

--- Maior lift, sem filtro de suporte ---


,antecedents,consequents,qtd_candidatos,confidence,lift
7337,"DS_COR_RACA=PARDA, DS_GENERO=MASCULINO, DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, SG_UF=MS",SG_UF_NASCIMENTO=MS,14,0.933333,6.727778
2492,"DS_COR_RACA=PRETA, SG_UF=MT",SG_UF_NASCIMENTO=MT,14,0.823529,6.550372
2544,"SG_UF=MS, cluster=Jovens com menor escolaridade e poucos bens",SG_UF_NASCIMENTO=MS,17,0.894737,6.449561
7481,"DS_GENERO=MASCULINO, DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, SG_UF=MS, cluster=Adultos com superior e bens",SG_UF_NASCIMENTO=MS,15,0.882353,6.360294
4637,"DS_GENERO=MASCULINO, SG_UF=MS, cluster=Adultos com superior e bens",SG_UF_NASCIMENTO=MS,15,0.882353,6.360294


--- Maior lift, exigindo qtd_candidatos >= 10 ---


,antecedents,consequents,qtd_candidatos,confidence,lift
7337,"DS_COR_RACA=PARDA, DS_GENERO=MASCULINO, DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, SG_UF=MS",SG_UF_NASCIMENTO=MS,14,0.933333,6.727778
2492,"DS_COR_RACA=PRETA, SG_UF=MT",SG_UF_NASCIMENTO=MT,14,0.823529,6.550372
2544,"SG_UF=MS, cluster=Jovens com menor escolaridade e poucos bens",SG_UF_NASCIMENTO=MS,17,0.894737,6.449561
7481,"DS_GENERO=MASCULINO, DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, SG_UF=MS, cluster=Adultos com superior e bens",SG_UF_NASCIMENTO=MS,15,0.882353,6.360294
4637,"DS_GENERO=MASCULINO, SG_UF=MS, cluster=Adultos com superior e bens",SG_UF_NASCIMENTO=MS,15,0.882353,6.360294


As regras com suporte baixo podem ter lift alto sem que sejam estáveis. Compare sempre o número de candidatos que sustenta cada regra. O filtro por quantidade pode ser redundante se o suporte mínimo do Apriori já exigir mais candidatos que esse piso.

### Filtrando regras para análise


In [56]:
suporte_pratico = 0.03  # mínimo de ceil(suporte_pratico * número de cestas) candidatos
confianca_pratica = 0.5

regras_simples = regras[
    regras["antecedents"].apply(lambda x: len(x) <= 2)
    & regras["consequents"].apply(lambda x: len(x) == 1)
]
regras_confiaveis = regras_simples[
    (regras_simples["support"] >= suporte_pratico)
    & (regras_simples["confidence"] >= confianca_pratica)
]

print(
    f"{len(regras_confiaveis)} regras com suporte >= {suporte_pratico} e confiança >= {confianca_pratica}"
)
formatar_regras(
    regras_confiaveis.sort_values("confidence", ascending=False)[colunas_exibir].head(
        15
    )
)

412 regras com suporte >= 0.03 e confiança >= 0.5


,antecedents,consequents,qtd_candidatos,confidence,lift
1863,"SG_UF_NASCIMENTO=MT, cluster=Adultos com superior e bens",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,30,1.0,1.534368
1838,"SG_UF_NASCIMENTO=GO, cluster=Adultos com superior e bens",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,61,1.0,1.534368
1850,"SG_UF_NASCIMENTO=MS, cluster=Adultos com superior e bens",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,27,1.0,1.534368
1883,"SG_UF=DF, cluster=Adultos com superior e bens",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,48,1.0,1.534368
1893,"SG_UF=MS, cluster=Adultos com superior e bens",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,31,1.0,1.534368
535,"SG_PARTIDO=PL, cluster=Adultos com superior e bens",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,21,1.0,1.534368
1897,"SG_UF=MT, cluster=Adultos com superior e bens",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,45,1.0,1.534368
1817,"SG_UF_NASCIMENTO=DF, cluster=Adultos com superior e bens",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,32,1.0,1.534368
1888,"SG_UF=GO, cluster=Adultos com superior e bens",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,79,1.0,1.534368
1790,"DS_COR_RACA=PARDA, cluster=Adultos com superior e bens",DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,72,1.0,1.534368


## Quais atributos estão associados ao cluster de um candidato?

Filtramos regras cujo **consequente é um único item de cluster**. Os seis grupos têm tamanhos diferentes: uma confiança elevada pode refletir apenas um grupo comum na base. Por isso selecionamos as regras de maior lift **dentro de cada cluster**, considerando também suporte e quantidade de candidatos. Um grupo pode não ter regras que passem pelos filtros; não forçamos sua presença.

Escolaridade foi usada para formar os clusters: regras que associam escolaridade a cluster podem refletir essa construção. Não são validação independente do agrupamento nem evidência de causalidade.


In [57]:
def regras_para_consequente(
    regras_completas,
    prefixo,
    n_por_valor=6,
    min_suporte=min_suporte,
    max_antecedentes=2,
):
    """
    Entre as regras cujo único item consequente comece com `prefixo` (ex.: 'cluster='),
    devolve as `n_por_valor` de MAIOR LIFT pra CADA valor distinto do consequente — sem piso
    de confiança fixo, porque um valor raro (poucos candidatos) não sustenta regras de
    confiança alta mesmo quando o lift é forte; ordenar só por confiança deixaria os valores
    mais comuns dominando a tabela inteira (ver célula anterior).
    """
    candidatas = regras_completas[
        regras_completas["consequents"].apply(
            lambda x: len(x) == 1 and next(iter(x)).startswith(prefixo)
        )
        & regras_completas["antecedents"].apply(lambda x: len(x) <= max_antecedentes)
        & (regras_completas["support"] >= min_suporte)
    ].copy()
    candidatas["alvo"] = candidatas["consequents"].apply(lambda x: next(iter(x)))
    resultado = (
        candidatas.sort_values("lift", ascending=False)
        .groupby("alvo", group_keys=False)
        .head(n_por_valor)
        .sort_values(["alvo", "lift"], ascending=[True, False])
    )
    return resultado.drop(columns="alvo")


regras_cluster = regras_para_consequente(regras, "cluster=")
print(
    f"{len(regras_cluster)} regras (até 6 por cluster, ordenadas por lift dentro de cada um)"
)
formatar_regras(regras_cluster[colunas_exibir])

36 regras (até 6 por cluster, ordenadas por lift dentro de cada um)


,antecedents,consequents,qtd_candidatos,confidence,lift
1561,"DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO, SG_UF=MT",cluster=Adultos com escolaridade intermediária e bens,15,0.500000,4.613333
1139,"DS_GENERO=MASCULINO, DS_GRAU_INSTRUCAO=SUPERIOR INCOMPLETO",cluster=Adultos com escolaridade intermediária e bens,17,0.425000,3.921333
1941,"DS_ESTADO_CIVIL=SOLTEIRO(A), DS_GRAU_INSTRUCAO=SUPERIOR INCOMPLETO",cluster=Adultos com escolaridade intermediária e bens,14,0.388889,3.588148
1044,"DS_GENERO=MASCULINO, DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO",cluster=Adultos com escolaridade intermediária e bens,27,0.380282,3.508732
1482,"DS_ESTADO_CIVIL=CASADO(A), DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO",cluster=Adultos com escolaridade intermediária e bens,19,0.380000,3.506133
243,DS_GRAU_INSTRUCAO=SUPERIOR INCOMPLETO,cluster=Adultos com escolaridade intermediária e bens,26,0.361111,3.331852
684,"DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, SG_PARTIDO=REPUBLICANOS",cluster=Adultos com superior e bens,19,0.655172,2.233396
534,"DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, SG_PARTIDO=PL",cluster=Adultos com superior e bens,21,0.617647,2.105477
608,"DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO, SG_PARTIDO=PSB",cluster=Adultos com superior e bens,17,0.566667,1.931691
1700,"DS_ESTADO_CIVIL=SOLTEIRO(A), DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO",cluster=Adultos com superior e bens,58,0.557692,1.901099


### Visualizando como grafo

Uma tabela com dezenas de regras é difícil de escanear — um grafo interativo ajuda a ver de relance quais itens aparecem mais como "explicação" (antecedentes) e quais aparecem mais como "resultado" (consequentes). Cada item vira um nó (`SG_PARTIDO=PCO`, `cluster=D`, ...); cada regra vira uma seta do antecedente pro consequente, com espessura proporcional à confiança. Os nós que começam com `cluster=` ficam destacados em vermelho e maiores, pra serem fáceis de achar no meio dos outros itens. Passe o mouse sobre uma seta pra ver confiança, lift e quantos candidatos sustentam aquela regra — e arraste os nós se a rede ficar embaralhada.


In [58]:
from pyvis.network import Network
from IPython.display import IFrame


def grafo_regras_html(regras_plot, arquivo_html, titulo, prefixo_destaque="cluster="):
    """
    Desenha as regras como uma rede interativa (pyvis) e grava em `arquivo_html`. Cada REGRA
    vira um nó de antecedente e um nó de consequente (quando o antecedente tem mais de um
    item, eles aparecem juntos no mesmo nó, separados por "+") ligados por UMA única aresta
    — nunca uma aresta por item isolado, pra não sugerir que um item sozinho sustenta a
    confiança/lift de uma regra que na verdade depende da combinação inteira (era isso que
    causava setas duplicadas entre o mesmo par de nós). Nós que contêm algum item começando
    com `prefixo_destaque` (por padrão, os clusters da Fase 2) ficam destacados em vermelho e
    maiores. Espessura da aresta = confiança da regra; passe o mouse pra ver confiança, lift
    e quantos candidatos a sustentam. Os controles de física (embaixo do grafo) deixam
    ajustar a repulsão/gravidade entre os nós ao vivo.

    Retorna um IFrame carregando o arquivo — se aparecer em branco/preto aqui embaixo (alguns
    ambientes bloqueiam o JavaScript de saídas de célula em notebooks "não confiáveis"), abra
    `arquivo_html` direto no navegador; o grafo é o mesmo.
    """
    print(titulo)
    net = Network(
        height="600px",
        width="100%",
        bgcolor="#222222",
        font_color="white",
        notebook=True,
        directed=True,
        cdn_resources="in_line",
    )

    nos_adicionados = set()
    for _, row in regras_plot.iterrows():
        antecedente = " + ".join(sorted(str(item) for item in row["antecedents"]))
        consequente = " + ".join(sorted(str(item) for item in row["consequents"]))

        for label, itemset in [
            (antecedente, row["antecedents"]),
            (consequente, row["consequents"]),
        ]:
            if label in nos_adicionados:
                continue
            destaque = any(str(item).startswith(prefixo_destaque) for item in itemset)
            net.add_node(
                label,
                label,
                title=label,
                color="#e41a1c" if destaque else "#1f78b4",
                size=30 if destaque else 12,
                font={"size": 22 if destaque else 14},
            )
            nos_adicionados.add(label)

        titulo_aresta = (
            f"Confiança: {row['confidence']:.2f} | Lift: {row['lift']:.2f} | "
            f"{row['qtd_candidatos']} candidatos"
        )
        net.add_edge(
            antecedente, consequente, value=row["confidence"], title=titulo_aresta
        )

    net.show_buttons(filter_=["physics"])
    net.save_graph(arquivo_html)
    print(
        f"Grafo salvo em: {arquivo_html} — se não aparecer abaixo, abra esse arquivo no navegador."
    )
    return IFrame(arquivo_html, width="100%", height=750)


grafo_regras_html(
    regras_cluster,
    "grafo_regras_cluster.html",
    "O que prediz o cluster de um candidato?",
)

O que prediz o cluster de um candidato?
Grafo salvo em: grafo_regras_cluster.html — se não aparecer abaixo, abra esse arquivo no navegador.


## O que caracteriza quem já é de um cluster?

Agora fixamos o **antecedente** (`cluster=X`) e procuramos os atributos associados a cada grupo. Ao inverter `A → B` para `B → A`, a confiança pode mudar, pois a população de referência muda. **O suporte e o lift são iguais nas duas direções** para os mesmos conjuntos de itens. Essas relações descrevem associações, não causas.


In [59]:
def regras_para_antecedente(
    regras_completas, item_fixo, n=8, min_suporte=min_suporte, max_consequentes=1
):
    """
    Entre as regras cujo antecedente é EXATAMENTE {item_fixo} (um único item), devolve as
    `n` de maior lift. Direção oposta de `regras_para_consequente`: lá fixamos o consequente; aqui fixamos o antecedente.
    A leitura é associativa, sem interpretação causal.
    """
    candidatas = regras_completas[
        regras_completas["antecedents"].apply(
            lambda x: len(x) == 1 and next(iter(x)) == item_fixo
        )
        & regras_completas["consequents"].apply(lambda x: len(x) <= max_consequentes)
        & (regras_completas["support"] >= min_suporte)
    ]
    return candidatas.sort_values("lift", ascending=False).head(n)


regras_por_cluster_fixo = {
    nome: regras_para_antecedente(regras, nome)
    for nome in (f"cluster={nome}" for nome in NOMES_CLUSTER_BISECTING.values())
}

for item_fixo, subset in regras_por_cluster_fixo.items():
    print(f"--- {item_fixo} ({len(subset)} regras) ---")
    display(formatar_regras(subset[colunas_exibir]))

--- cluster=Adultos com superior e bens (8 regras) ---


,antecedents,consequents,qtd_candidatos,confidence,lift
92,cluster=Adultos com superior e bens,SG_PARTIDO=PSB,17,0.083744,1.704433
67,cluster=Adultos com superior e bens,SG_PARTIDO=PL,21,0.103448,1.590805
231,cluster=Adultos com superior e bens,DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,203,1.000000,1.534368
25,cluster=Adultos com superior e bens,SG_PARTIDO=MDB,15,0.073892,1.420361
131,cluster=Adultos com superior e bens,SG_PARTIDO=REPUBLICANOS,19,0.093596,1.378053
347,cluster=Adultos com superior e bens,SG_UF_NASCIMENTO=DF,32,0.157635,1.283338
372,cluster=Adultos com superior e bens,SG_UF_NASCIMENTO=SP,15,0.073892,1.189140
367,cluster=Adultos com superior e bens,SG_UF_NASCIMENTO=MT,30,0.147783,1.175471


--- cluster=Mais velhos com superior e bens (8 regras) ---


,antecedents,consequents,qtd_candidatos,confidence,lift
121,cluster=Mais velhos com superior e bens,SG_PARTIDO=PT,17,0.092391,2.131159
235,cluster=Mais velhos com superior e bens,DS_GRAU_INSTRUCAO=SUPERIOR COMPLETO,172,0.934783,1.434301
111,cluster=Mais velhos com superior e bens,SG_PARTIDO=PSDB,15,0.081522,1.375928
317,cluster=Mais velhos com superior e bens,DS_COR_RACA=BRANCA,125,0.679348,1.246973
395,cluster=Mais velhos com superior e bens,SG_UF=DF,56,0.304348,1.246205
265,cluster=Mais velhos com superior e bens,DS_ESTADO_CIVIL=CASADO(A),117,0.635870,1.225687
277,cluster=Mais velhos com superior e bens,DS_ESTADO_CIVIL=DIVORCIADO(A),31,0.168478,1.165870
187,cluster=Mais velhos com superior e bens,DS_GENERO=MASCULINO,121,0.657609,1.075804


--- cluster=Alta escolaridade sem bens declarados (8 regras) ---


...

,antecedents,consequents,qtd_candidatos,confidence,lift
32,cluster=Jovens com menor escolaridade e poucos bens,SG_PARTIDO=MISSÃO,14,0.155556,3.472401
204,cluster=Jovens com menor escolaridade e poucos bens,DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO,39,0.433333,2.701502
340,cluster=Jovens com menor escolaridade e poucos bens,DS_COR_RACA=PRETA,20,0.222222,2.261438
295,cluster=Jovens com menor escolaridade e poucos bens,DS_ESTADO_CIVIL=SOLTEIRO(A),61,0.677778,2.151478
246,cluster=Jovens com menor escolaridade e poucos bens,DS_GRAU_INSTRUCAO=SUPERIOR INCOMPLETO,19,0.211111,2.029012
362,cluster=Jovens com menor escolaridade e poucos bens,SG_UF_NASCIMENTO=MS,18,0.200000,1.441667
162,cluster=Jovens com menor escolaridade e poucos bens,DS_GENERO=FEMININO,47,0.522222,1.343412
330,cluster=Jovens com menor escolaridade e poucos bens,DS_COR_RACA=PARDA,40,0.444444,1.331409


--- cluster=Adultos com escolaridade intermediária e bens (8 regras) ---


,antecedents,consequents,qtd_candidatos,confidence,lift
242,cluster=Adultos com escolaridade intermediária e bens,DS_GRAU_INSTRUCAO=SUPERIOR INCOMPLETO,26,0.346667,3.331852
203,cluster=Adultos com escolaridade intermediária e bens,DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO,37,0.493333,3.075556
293,cluster=Adultos com escolaridade intermediária e bens,DS_ESTADO_CIVIL=SOLTEIRO(A),34,0.453333,1.439021
379,cluster=Adultos com escolaridade intermediária e bens,SG_UF=MT,20,0.266667,1.337198
326,cluster=Adultos com escolaridade intermediária e bens,DS_COR_RACA=PARDA,32,0.426667,1.278153
376,cluster=Adultos com escolaridade intermediária e bens,SG_UF=GO,36,0.480000,1.267786
352,cluster=Adultos com escolaridade intermediária e bens,SG_UF_NASCIMENTO=GO,24,0.320000,1.177872
183,cluster=Adultos com escolaridade intermediária e bens,DS_GENERO=MASCULINO,51,0.680000,1.112435


--- cluster=Mais velhos com menor escolaridade e bens (5 regras) ---


,antecedents,consequents,qtd_candidatos,confidence,lift
207,cluster=Mais velhos com menor escolaridade e bens,DS_GRAU_INSTRUCAO=ENSINO MÉDIO COMPLETO,21,0.617647,3.850556
184,cluster=Mais velhos com menor escolaridade e bens,DS_GENERO=MASCULINO,29,0.852941,1.395355
262,cluster=Mais velhos com menor escolaridade e bens,DS_ESTADO_CIVIL=CASADO(A),20,0.588235,1.133869
392,cluster=Mais velhos com menor escolaridade e bens,SG_UF=GO,14,0.411765,1.087562
314,cluster=Mais velhos com menor escolaridade e bens,DS_COR_RACA=BRANCA,20,0.588235,1.079732


### Visualizando como grafo (antecedente fixo)

Os seis clusters são os pontos de partida das regras; os atributos associados aparecem como consequentes.


In [60]:
regras_antecedente_fixo = pd.concat(regras_por_cluster_fixo.values())
grafo_regras_html(
    regras_antecedente_fixo,
    "grafo_regras_antecedente_fixo.html",
    "O que caracteriza quem já é de um cluster?",
)

O que caracteriza quem já é de um cluster?
Grafo salvo em: grafo_regras_antecedente_fixo.html — se não aparecer abaixo, abra esse arquivo no navegador.


## E quando a eleição já tiver ocorrido?

Esta seção só roda depois que a Fase 0 for rodada de novo com o resultado oficial (`DS_SIT_TOT_TURNO` deixa de vir toda com o mesmo valor `#NULO`) — por enquanto, ela se anuncia e não faz nada. Repare que não precisamos escrever nenhum código novo: como `DS_SIT_TOT_TURNO` já entrou na cesta lá na preparação (quando aplicável), a mesma `regras_para_consequente` e o mesmo `grafo_regras_html` servem pra essa pergunta também.


In [61]:
if usar_resultado_eleicao:
    regras_eleicao = regras_para_consequente(regras, "DS_SIT_TOT_TURNO=")
    print(f"{len(regras_eleicao)} regras apontando para o resultado da eleição")
    display(formatar_regras(regras_eleicao[colunas_exibir].head(15)))
    grafo_regras_html(
        regras_eleicao.head(20),
        "grafo_regras_eleicao.html",
        "O que prediz ser eleito?",
        prefixo_destaque="DS_SIT_TOT_TURNO=",
    )
else:
    print(
        "DS_SIT_TOT_TURNO ainda não tem informação real (a eleição de 2026 não ocorreu) — "
        "essa seção fica pronta pra usar assim que a Fase 0 for rodada de novo com o resultado oficial. "
        "Por enquanto, pulamos."
    )

DS_SIT_TOT_TURNO ainda não tem informação real (a eleição de 2026 não ocorreu) — essa seção fica pronta pra usar assim que a Fase 0 for rodada de novo com o resultado oficial. Por enquanto, pulamos.


---

## Desafio: Regras com variáveis \_Faixa (Parte 2 do desafio_gastos_campanha)

O `04_desafio_gastos_campanha.ipynb` criou versões categóricas por quartil de todas as variáveis-driver:

- `IDADE_Faixa`, `ANOS_ESTUDO_Faixa`, `Total_Bens_Log_Faixa`, `Total_Gasto_Campanha_Log_Faixa`

Com essas colunas, o Apriori consegue usar `IDADE_Faixa=Q1` (os mais jovens) como item de verdade na cesta — algo que não era possível antes, já que o Apriori só enxerga itens categóricos.

Esta seção carrega o `df_cluster` enriquecido (salvo pelo desafio) e inclui as colunas `_Faixa` na cesta, gerando regras novas como:

> _"IDADE_Faixa=Q1 + DS_GENERO=MASCULINO → cluster=Jovens com menor escolaridade e poucos bens"_


In [62]:
# Tenta carregar o df_cluster com colunas _Faixa gerado pelo 04_desafio_gastos_campanha.ipynb
import os

arquivo_desafio = f"dados/candidatos_{nome_cargo}_COM_GASTO_FAIXAS_{ANO_ELEICAO}.csv"

if not os.path.exists(arquivo_desafio):
    print(f"Arquivo não encontrado: {arquivo_desafio}")
    print("Rode o 04_desafio_gastos_campanha.ipynb primeiro para gerar esse arquivo.")
else:
    df_com_faixas = pd.read_csv(arquivo_desafio)
    print(f"Carregado: {arquivo_desafio} -> {df_com_faixas.shape}")

    # Reutiliza os clusters da Fase 2 por candidato; não retreina em outra base.
    df_com_faixas = df_com_faixas.drop(
        columns=["cluster", "cluster_bisecting"], errors="ignore"
    )
    df_com_faixas = df_com_faixas.merge(
        df_cluster[["SQ_CANDIDATO", "cluster_bisecting", "cluster"]],
        on="SQ_CANDIDATO",
        how="inner",
        validate="one_to_one",
    )
    print(
        f"{len(df_com_faixas)} candidatos com faixas e cluster da mesma partição da Fase 2"
    )

    # Colunas _Faixa disponíveis
    colunas_faixa = [c for c in df_com_faixas.columns if c.endswith("_Faixa")]
    print(f"Colunas _Faixa disponíveis: {colunas_faixa}")

    # Monta cesta enriquecida (inclui as _Faixa como itens)
    colunas_apriori_f = [
        "SG_PARTIDO",
        "DS_GENERO",
        "DS_GRAU_INSTRUCAO",
        "DS_ESTADO_CIVIL",
        "DS_COR_RACA",
        "SG_UF_NASCIMENTO",
        "cluster",
    ] + colunas_faixa

    if df_cluster["SG_UF"].nunique() > 1:
        colunas_apriori_f.append("SG_UF")

    cesta_f_base = df_com_faixas[["SQ_CANDIDATO"] + colunas_apriori_f].copy()
    # Converter categoricals para str (necessário para pd.get_dummies tratar como categorias)
    for c in colunas_faixa:
        cesta_f_base[c] = cesta_f_base[c].astype(str)

    cesta_f = pd.get_dummies(cesta_f_base, columns=colunas_apriori_f, prefix_sep="=")
    cesta_f = cesta_f.set_index("SQ_CANDIDATO")
    print(f"{cesta_f.shape[0]} candidatos x {cesta_f.shape[1]} itens (com _Faixa)")

    # Apriori na cesta enriquecida
    itens_f = apriori(cesta_f, min_support=min_suporte, use_colnames=True)
    regras_f = association_rules(itens_f, metric="lift", min_threshold=1)
    regras_f["qtd_candidatos"] = (
        (regras_f["support"] * len(cesta_f)).round().astype(int)
    )

    # Regras novas: envolvem alguma _Faixa como antecedente ou consequente
    prefixos_faixa = tuple(f"{c}=" for c in colunas_faixa)

    def tem_faixa(itemset):
        return any(str(i).startswith(prefixos_faixa) for i in itemset)

    regras_novas = regras_f[
        regras_f["antecedents"].apply(tem_faixa)
        & regras_f["consequents"].apply(lambda x: len(x) == 1)
        & (regras_f["qtd_candidatos"] >= 8)
        & (regras_f["lift"] >= 1.2)
    ].sort_values("lift", ascending=False)

    print(
        f"\n{len(regras_novas)} regras novas com antecedente _Faixa (qtd >= 8, lift >= 1.2):"
    )
    print("--- Top 15 por lift ---")
    display(
        formatar_regras(
            regras_novas[
                ["antecedents", "consequents", "qtd_candidatos", "confidence", "lift"]
            ].head(15)
        )
    )

    # Uma regra exemplar: destacar a primeira que envolve _Faixa → cluster
    regras_faixa_cluster = regras_novas[
        regras_novas["consequents"].apply(
            lambda x: next(iter(x)).startswith("cluster=")
        )
    ]
    if not regras_faixa_cluster.empty:
        r = regras_faixa_cluster.iloc[0]
        ant = ", ".join(sorted(str(i) for i in r["antecedents"]))
        con = ", ".join(sorted(str(i) for i in r["consequents"]))
        print(f"\nExemplo de regra nova (não existia sem as _Faixa):")
        print(f"  {ant} → {con}")
        print(
            f"  Confiança: {r['confidence']:.2f} | Lift: {r['lift']:.2f} | {r['qtd_candidatos']} candidatos"
        )

Arquivo não encontrado: dados/candidatos_DEPUTADO_FEDERAL_COM_GASTO_FAIXAS_2026.csv
Rode o 04_desafio_gastos_campanha.ipynb primeiro para gerar esse arquivo.


---

## Fechamento e próximos passos

Resumindo o roteiro: transformamos cada candidato numa "cesta" de atributos categóricos (partido, gênero, escolaridade, cor/raça, UF de nascimento, e o cluster da Fase 2), minerar itemsets frequentes com o Apriori, geramos regras `A → B` com suporte/confiança/lift, e aprendemos — na prática, não só na teoria — que suporte baixo infla lift artificialmente numa base pequena como esta.

**Exercícios sugeridos:**

- Agrupar `DS_OCUPACAO` em categorias maiores e incluir na cesta.
- Variar `min_suporte` (célula da seção "Itens frequentes") e ver como o número de regras muda.
- Replicar para `SENADOR` ou `DEPUTADO FEDERAL`, na sua UF — só muda a célula de configuração, do mesmo jeito que nas fases anteriores.
- Depois que a eleição ocorrer: rodar a Fase 0 de novo, e conferir se a seção "E quando a eleição já tiver ocorrido?" acima encontra regras fortes ligadas a `DS_SIT_TOT_TURNO=ELEITO`.

Próxima etapa do pipeline: `04_deteccao_anomalias.ipynb`.
